In [29]:
import pandas as pd
import numpy as np

In [30]:
df = pd.read_csv("../data/raw/results.csv")

df["date"] = pd.to_datetime(df["date"])

completed_matches = df[
    df["home_score"].notna() &
    df["away_score"].notna()
].copy()

completed_matches = completed_matches.sort_values("date")

In [31]:
def get_tournament_weight(tournament):

    if tournament == "FIFA World Cup":
        return 60

    major_continental = {"UEFA Euro", "Copa América", "African Cup of Nations", "AFC Asian Cup", "Gold Cup", "Oceania Nations Cup"}

    if tournament in major_continental:
        return 50

    if "qualification" in tournament.lower():
        return 40
    
    if "Nations League" in tournament:
        return 35
    
    if tournament == "Friendly":
        return 20
    
    return 30

In [32]:
#Test
print(get_tournament_weight("Friendly"))

print(get_tournament_weight("FIFA World Cup"))

print(get_tournament_weight("FIFA World Cup qualification"))

print(get_tournament_weight("UEFA Euro"))

print(get_tournament_weight("UEFA Nations League"))

20
60
40
50
35


In [33]:
def get_goal_difference_multiplier(home_score, away_score):

    goal_diff = abs(home_score - away_score)

    if goal_diff <= 1:
        return 1.0

    elif goal_diff == 2:
        return 1.5
    
    elif goal_diff == 3:
        return 1.75
    
    else:
        return (1.75 + (goal_diff - 3) / 8)

In [34]:
print(get_goal_difference_multiplier(1,0))

print(get_goal_difference_multiplier(2,0))

print(get_goal_difference_multiplier(3,0))

print(get_goal_difference_multiplier(5,0))

1.0
1.5
1.75
2.0


In [35]:
def expected_score(rating_a, rating_b):

    return (
        1 /
        (
            1 +
            10 ** (
                (rating_b - rating_a) / 400
            )
        )
    )

In [36]:
expected_score(1500, 1500)

0.5

In [37]:
expected_score(1600, 1500)

0.6400649998028851

In [38]:
def apply_home_advantage(home_rating, neutral):
    if neutral:
        return home_rating
    
    return home_rating + 100

In [39]:
print(apply_home_advantage(1500, False))

print(apply_home_advantage(1500, True))

1600
1500


In [40]:
def update_elo(home_rating, away_rating, home_score, away_score, tournament, neutral):

    #Apply Home Advantage if any
    adjusted_home_rating = apply_home_advantage(home_rating, neutral)

    #Expected Results
    expected_home = expected_score(adjusted_home_rating, away_rating)
    expected_away = 1 - expected_home

    #Actual Results
    if home_score > away_score:
        actual_home = 1
        actual_away = 0
    
    elif home_score < away_score:
        actual_home = 0
        actual_away = 1
    
    else:
        actual_home = 0.5
        actual_away = 0.5
    
    #Considering Tournament Weightage
    k = get_tournament_weight(tournament)

    #Goal Difference Multiplier
    multiplier = get_goal_difference_multiplier(home_score, away_score)

    #Final Adjustment
    adjustment = k * multiplier

    new_home_rating = (home_rating + adjustment * (actual_home - expected_home))
    new_away_rating = (away_rating + adjustment * (actual_away - expected_away))

    return (new_home_rating, new_away_rating)

In [41]:
update_elo(1500, 1500, 1, 0, "Friendly", False)

(1507.1987000039423, 1492.8012999960577)

In [42]:
update_elo(1200, 1800, 1, 0, "FIFA World Cup", True)

(1258.1607941980972, 1741.8392058019028)

In [43]:
elo_ratings = {}

elo_history = []

In [44]:
for idx, (_, match) in enumerate(completed_matches.iterrows()):

    if idx % 1000 == 0:
        print(f"Processed : {idx}")
    
    home_team = match["home_team"]
    away_team = match["away_team"]

    home_score = match["home_score"]
    away_score = match["away_score"]

    tournament = match["tournament"]
    neutral = match["neutral"]

    date = match["date"]

    #Initialise teams
    if home_team not in elo_ratings:
        elo_ratings[home_team] = 1500

    if away_team not in elo_ratings:
        elo_ratings[away_team] = 1500
    
    home_rating = elo_ratings[home_team]
    away_rating = elo_ratings[away_team]

    new_home_rating, new_away_rating = update_elo(home_rating, away_rating, home_score, away_score, tournament, neutral)

    elo_ratings[home_team] = new_home_rating
    elo_ratings[away_team] = new_away_rating

    elo_history.append({"date": date, "team": home_team, "elo": new_home_rating})
    elo_history.append({"date": date, "team": away_team, "elo": new_away_rating})

Processed : 0
Processed : 1000
Processed : 2000
Processed : 3000
Processed : 4000
Processed : 5000
Processed : 6000
Processed : 7000
Processed : 8000
Processed : 9000
Processed : 10000
Processed : 11000
Processed : 12000
Processed : 13000
Processed : 14000
Processed : 15000
Processed : 16000
Processed : 17000
Processed : 18000
Processed : 19000
Processed : 20000
Processed : 21000
Processed : 22000
Processed : 23000
Processed : 24000
Processed : 25000
Processed : 26000
Processed : 27000
Processed : 28000
Processed : 29000
Processed : 30000
Processed : 31000
Processed : 32000
Processed : 33000
Processed : 34000
Processed : 35000
Processed : 36000
Processed : 37000
Processed : 38000
Processed : 39000
Processed : 40000
Processed : 41000
Processed : 42000
Processed : 43000
Processed : 44000
Processed : 45000
Processed : 46000
Processed : 47000
Processed : 48000
Processed : 49000


In [45]:
elo_df = pd.DataFrame(elo_history)

elo_df.head()

,date,team,elo
0,1872-11-30,Scotland,1497.198700
1,1872-11-30,England,1502.801300
2,1873-03-08,England,1513.377469
3,1873-03-08,Scotland,1486.622531
4,1874-03-07,Scotland,1494.545054


In [46]:
elo_df.tail()

,date,team,elo
98795,2026-06-09,Central African Republic,1361.545513
98796,2026-06-09,Ethiopia,1391.946659
98797,2026-06-09,Malawi,1437.620917
98798,2026-06-09,Iraq,1737.905802
98799,2026-06-09,Venezuela,1822.280945


In [48]:
len(elo_ratings)

336

In [49]:
sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True)[:20]

[('Spain', 2213.2631922856417),
 ('Argentina', 2188.479269061217),
 ('France', 2121.4889497833856),
 ('England', 2082.854062026291),
 ('Brazil', 2067.452599576456),
 ('Colombia', 2062.769426962693),
 ('Portugal', 2033.2561239100087),
 ('Ecuador', 2026.6299428217587),
 ('Netherlands', 2009.7814809483577),
 ('Germany', 2004.1537741400791),
 ('Japan', 1992.7455307250605),
 ('Morocco', 1984.2319130705296),
 ('Mexico', 1977.7575970507135),
 ('Uruguay', 1974.076992701208),
 ('Croatia', 1965.3169408579997),
 ('Turkey', 1964.760828624828),
 ('Norway', 1963.0467607507553),
 ('Belgium', 1962.2792806874647),
 ('Switzerland', 1956.7166944133248),
 ('Italy', 1922.49972968304)]

In [50]:
elo_df.to_csv("../data/processed/elo_history.csv", index=False)

In [51]:
def get_elo(team, date, elo_df):

    team_history = elo_df[(elo_df["team"] == team) & (elo_df["date"] < date)]

    if len(team_history) == 0:
        return np.nan
    
    return team_history.iloc[-1]["elo"]

In [52]:
print(
    get_elo(
        "Argentina",
        pd.Timestamp("2026-01-01"),
        elo_df
    )
)

print(
    get_elo(
        "Brazil",
        pd.Timestamp("2026-01-01"),
        elo_df
    )
)

2184.570942866439
2053.092251076258
